In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv(r"C:\Users\konra\Desktop\vscode\projekt_zespolowy\smart-shopping-list-ML\ml\data\kaggle_prepared.csv")

In [3]:
df.head()

,order_id,user_id,days_since_prior_order,off_product_id,aisle_id
0,2539329,1,0.0,"['5902802802576', '25026290', '5908230536007',...","[[297, 2142, 2005], [1430, 1033, 2059, 2058, 1..."
1,2398795,1,15.0,"['5902802802576', '8711327672499', '5908230536...","[[297, 2142, 2005], [277, 989, 976, 780, 777, ..."
2,473747,1,21.0,"['5902802802576', '5908230536007', '8711327672...","[[297, 2142, 2005], [1801], [277, 989, 976, 78..."
3,2254736,1,29.0,"['5902802802576', '5908230536007', '8711327672...","[[297, 2142, 2005], [1801], [277, 989, 976, 78..."
4,431534,1,28.0,"['5902802802576', '5908230536007', '8711327672...","[[297, 2142, 2005], [1801], [277, 989, 976, 78..."


In [4]:
number_of_users = df['user_id'].nunique()
print(f"Liczba unikalnych użytkowników: {number_of_users}")

Liczba unikalnych użytkowników: 206209


In [5]:
import ast
n_samples =5
df["off_product_id"] = df["off_product_id"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else []
    )
min_orders = 8+1
valid_users = [uid for uid, grp in df.groupby("user_id") if len(grp) >= min_orders]
step = max(1, len(valid_users) // n_samples)
chosen = valid_users[::step][:n_samples]
print(f"[EVAL] Wybrano {len(chosen)} użytkowników do ewaluacji  (co {step}-ty z {len(valid_users)})")
for i, uid in enumerate(chosen, 1):
        print(f"\n{'─'*55}")
        print(f"[EVAL {i}/{len(chosen)}] user_id={uid}")

        user_df = df[df["user_id"] == uid].sort_values("order_id").reset_index(drop=True)
        print(user_df)

[EVAL] Wybrano 5 użytkowników do ewaluacji  (co 23440-ty z 117200)

───────────────────────────────────────────────────────
[EVAL 1/5] user_id=1
    order_id  user_id  days_since_prior_order  \
0     431534        1                    28.0   
1     473747        1                    21.0   
2     550135        1                    20.0   
3    1187899        1                    14.0   
4    2254736        1                    29.0   
5    2295261        1                     0.0   
6    2398795        1                    15.0   
7    2539329        1                     0.0   
8    2550362        1                    30.0   
9    3108588        1                    14.0   
10   3367565        1                    19.0   

                                       off_product_id  \
0   [5902802802576, 5908230536007, 8711327672499, ...   
1   [5902802802576, 5908230536007, 8711327672499, ...   
2   [5902802802576, 8711327672499, 5908230536007, ...   
3   [5902802802576, 2228326001442, 590

In [35]:
def _parse_categories(raw) -> list[str]:
    if isinstance(raw, list):
        return [str(c).strip().lower() for c in raw if c]
    if not isinstance(raw, str) or not raw.strip():
        return []
    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, list):
            return [str(c).strip().lower() for c in parsed if c]
    except Exception:
        pass
    return [raw.strip().lower()]
df2 = pd.read_csv(r'C:\Users\konra\Desktop\vscode\projekt_zespolowy\smart-shopping-list-ML\prepare_data_for_model\OpenFoodData.csv', usecols=["Id", "Name", "Categories"])
df2["Id"] = df2["Id"].astype(str)
df2 = df2.dropna(subset=["Name"])
_product_cache = {}
for _, row in df2.iterrows():
    pid = row["Id"]
    _product_cache[pid] = {
        "name": str(row["Name"]).strip(),
        "categories": _parse_categories(row.get("Categories", "")),
    }

recalls = []
for i, uid in enumerate(chosen, 1):
        print(f"\n{'─'*55}")
        print(f"[EVAL {i}/{len(chosen)}] user_id={uid}")
        id2name: dict[str, str] = {pid: info["name"] for pid, info in _product_cache.items()}
        user_df = df[df["user_id"] == uid].sort_values("order_id").reset_index(drop=True)
        history_rows = user_df.iloc[:-1]
        target_rows = user_df.iloc[-1]
        target = []
        for x in target_rows['off_product_id']:
            target.append(id2name[x])
        baskets = []
        history_baskets = {'history_baskets': []}
        for _, row in history_rows.iterrows():
            products = []
            for x in row["off_product_id"]:
                name = [id2name[str(x)]]
                products.append({'name':name, 'id':x})
            days = int(row["days_since_prior_order"]) if pd.notna(row.get("days_since_prior_order")) else 0
            history_baskets['history_baskets'].append({"date": days, "products": products})
            #baskets.append({"days": days, "products": names})
        print(history_baskets)
        import rag_pipeline
        import importlib
        importlib.reload(rag_pipeline)
        from rag_pipeline import stage1, stage2, stage3
        res_stage1 = stage1(history_baskets)
        print(res_stage1)
        res_stage2 = stage2(res_stage1)
        print(res_stage2)
        res_stage3 = stage3(res_stage1, res_stage2, final_k=10)
        print(res_stage3) 
        recall=0
        for r in res_stage3['recommended_ids']:
            if r in target:
                recall+=1
        recalls.append(recall/len(target))
        print(recall/len(target))
res = sum(recalls)/len(recalls)
print(f"Recall@10: {res}")


───────────────────────────────────────────────────────
[EVAL 1/5] user_id=1
{'history_baskets': [{'date': 28, 'products': [{'name': ['Soda oczyszczona spożywcza bez antyzbrylaczy – 1000g'], 'id': '5902802802576'}, {'name': ['Beef Jerky – Tarczyński'], 'id': '5908230536007'}, {'name': ['Pistachio Delight – Carte d’Or'], 'id': '8711327672499'}, {'name': ['Ser włoszczowski – Auchan'], 'id': '2228326001442'}, {'name': ['Jabłka – La-Sad – 1,5 kg'], 'id': '20355944'}, {'name': ['Suszone jabłko o smaku ananasa – Crispy natural – 18g'], 'id': '5900238965421'}, {'name': ['Milk chocolate-covered banana – Biedronka – 180 g'], 'id': '5907554477454'}]}, {'date': 21, 'products': [{'name': ['Soda oczyszczona spożywcza bez antyzbrylaczy – 1000g'], 'id': '5902802802576'}, {'name': ['Beef Jerky – Tarczyński'], 'id': '5908230536007'}, {'name': ['Pistachio Delight – Carte d’Or'], 'id': '8711327672499'}, {'name': ['Ser włoszczowski – Auchan'], 'id': '2228326001442'}, {'name': ['Almond Butter'], 'id': '59